[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_09/notebook_9_1_dataset_bias_measurement.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 9.1: Measuring Dataset Bias

**Chapter 9: Fairness and Bias in Healthcare AI**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Identify sources of representation bias in medical datasets
2. Measure demographic imbalance quantitatively
3. Visualize distribution disparities across protected groups
4. Apply statistical tests to detect significant underrepresentation
5. Assess label quality differences across demographic groups
6. Understand how data collection practices lead to biased datasets

## Clinical Context

**The Foundation of Fair AI**: Before we can build fair AI systems, we must understand whether our training data fairly represents the populations we intend to serve. Biased data leads to biased models.

**Real-world examples**:
- Dermatology datasets: 90% light skin, 10% dark skin (doesn't match global population)
- Chest X-ray datasets: Predominantly from US/European academic centers
- EHR datasets: Underrepresent uninsured and rural populations

**Why this matters**: Priya's melanoma was nearly missed because the AI was trained on predominantly light-skin images. Dataset bias is the root cause of many fairness failures.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print("\nThis notebook demonstrates how to measure bias in medical datasets.")

## Part 1: Simulated Medical Dataset with Realistic Bias

We'll create a synthetic medical imaging dataset (inspired by dermatology) with realistic representation bias:
- **Skin tone distribution** (Fitzpatrick scale I-VI)
- **Disease prevalence** (melanoma)
- **Geographic source** of data
- **Data quality** variations

This simulates what real datasets look like before bias is addressed.

In [ ]:
def create_biased_dermatology_dataset(n_samples=10000, bias_level='high'):
    """
    Create synthetic dermatology dataset with realistic representation bias.

    Args:
        n_samples: Total number of samples
        bias_level: 'high', 'medium', 'low' - controls degree of representation bias

    Returns:
        DataFrame with patient demographics and outcomes
    """

    # Define skin tone distribution based on bias level
    # Fitzpatrick types: I-II (very light), III (light), IV (medium), V-VI (dark)
    if bias_level == 'high':
        # Typical of historical dermatology datasets
        skin_tone_probs = [0.40, 0.35, 0.15, 0.07, 0.03]  # 90% types I-III
    elif bias_level == 'medium':
        skin_tone_probs = [0.30, 0.30, 0.20, 0.12, 0.08]  # 80% types I-III
    else:  # low bias
        skin_tone_probs = [0.20, 0.20, 0.20, 0.20, 0.20]  # Equal representation

    skin_tone_labels = ['I-II (Very Light)', 'III (Light)', 'IV (Medium)',
                       'V (Medium-Dark)', 'VI (Dark)']

    # Generate skin tones
    skin_tones = np.random.choice(skin_tone_labels, size=n_samples, p=skin_tone_probs)

    # Geographic source (affects data quality and demographics)
    # High bias: predominantly wealthy Western academic centers
    if bias_level == 'high':
        geo_probs = [0.50, 0.30, 0.15, 0.05]  # US, Europe, Asia, Other
    elif bias_level == 'medium':
        geo_probs = [0.40, 0.25, 0.20, 0.15]
    else:
        geo_probs = [0.25, 0.25, 0.25, 0.25]

    geography = np.random.choice(['North America', 'Europe', 'Asia', 'Other'],
                                 size=n_samples, p=geo_probs)

    # Age distribution
    ages = np.random.normal(55, 15, n_samples).clip(18, 90).astype(int)

    # Sex
    sex = np.random.choice(['Male', 'Female'], size=n_samples, p=[0.48, 0.52])

    # Disease label (melanoma vs benign)
    # True prevalence ~2% for screening populations
    # But varies slightly by skin tone (biological + social factors)
    base_melanoma_rate = 0.02
    melanoma = []

    for tone in skin_tones:
        if 'Very Light' in tone or 'Light' in tone:
            rate = base_melanoma_rate * 1.2  # Slightly higher in lighter skin
        else:
            rate = base_melanoma_rate * 0.8  # Slightly lower in darker skin

        melanoma.append(np.random.random() < rate)

    melanoma = np.array(melanoma)

    # Image quality (proxy for data quality)
    # Lower quality for underrepresented groups (fewer specialized images)
    image_quality = []
    for tone, geo in zip(skin_tones, geography):
        base_quality = 0.8

        # Quality penalty for darker skin (fewer dermoscopic images collected)
        if 'Dark' in tone:
            base_quality -= 0.15

        # Quality boost for wealthy academic centers
        if geo in ['North America', 'Europe']:
            base_quality += 0.1

        # Add noise
        quality = np.clip(np.random.normal(base_quality, 0.1), 0, 1)
        image_quality.append(quality)

    image_quality = np.array(image_quality)

    # Label quality (confidence in diagnosis)
    # Biopsy-confirmed vs clinical diagnosis
    # More biopsy confirmation in academic centers
    biopsy_confirmed = []
    for geo, has_melanoma in zip(geography, melanoma):
        if geo in ['North America', 'Europe']:
            prob_biopsy = 0.85 if has_melanoma else 0.40
        else:
            prob_biopsy = 0.70 if has_melanoma else 0.25

        biopsy_confirmed.append(np.random.random() < prob_biopsy)

    biopsy_confirmed = np.array(biopsy_confirmed)

    # Create DataFrame
    df = pd.DataFrame({
        'patient_id': range(n_samples),
        'skin_tone': skin_tones,
        'geography': geography,
        'age': ages,
        'sex': sex,
        'melanoma': melanoma.astype(int),
        'image_quality': image_quality,
        'biopsy_confirmed': biopsy_confirmed.astype(int)
    })

    return df

# Create three datasets with different bias levels
print("Creating synthetic dermatology datasets...\n")

df_high_bias = create_biased_dermatology_dataset(10000, bias_level='high')
df_medium_bias = create_biased_dermatology_dataset(10000, bias_level='medium')
df_low_bias = create_biased_dermatology_dataset(10000, bias_level='low')

print("✓ Created 3 datasets:")
print(f"  1. High bias (realistic): {len(df_high_bias)} samples")
print(f"  2. Medium bias: {len(df_medium_bias)} samples")
print(f"  3. Low bias (aspirational): {len(df_low_bias)} samples")
print("\nWe'll focus on the HIGH BIAS dataset (most realistic)")

df = df_high_bias.copy()
print("\nDataset preview:")
display(df.head(10))

## Part 2: Measuring Representation Bias

**Representation bias**: When the dataset demographics don't match the population we want to serve.

### Key Questions:
1. What is the distribution of protected attributes?
2. How does this compare to the real-world population?
3. Is the imbalance statistically significant?
4. Which groups are most underrepresented?

In [ ]:
def measure_representation_bias(df, attribute='skin_tone',
                               expected_distribution=None):
    """
    Measure representation bias for a demographic attribute.

    Args:
        df: Dataset
        attribute: Column name of protected attribute
        expected_distribution: Dict mapping categories to expected proportions

    Returns:
        Dictionary with bias metrics
    """
    # Observed distribution
    observed_counts = df[attribute].value_counts()
    observed_props = df[attribute].value_counts(normalize=True)

    print(f"="*60)
    print(f"REPRESENTATION ANALYSIS: {attribute.upper()}")
    print("="*60)
    print("\nObserved Distribution:")
    for cat in observed_counts.index:
        count = observed_counts[cat]
        pct = observed_props[cat] * 100
        print(f"  {cat:30s}: {count:5d} ({pct:5.1f}%)")

    # If expected distribution provided, compute bias metrics
    if expected_distribution:
        print("\n" + "-"*60)
        print("BIAS ANALYSIS (vs Expected Population Distribution)")
        print("-"*60)

        total = len(df)
        chi2_stat = 0
        max_disparity = 0
        disparities = {}

        for cat, expected_prop in expected_distribution.items():
            if cat in observed_props.index:
                observed_prop = observed_props[cat]
                observed_count = observed_counts[cat]
                expected_count = total * expected_prop

                # Compute disparity
                disparity = observed_prop - expected_prop
                disparity_pct = (disparity / expected_prop) * 100 if expected_prop > 0 else 0
                disparities[cat] = disparity

                # Chi-square contribution
                chi2_contrib = ((observed_count - expected_count) ** 2) / expected_count
                chi2_stat += chi2_contrib

                if abs(disparity) > abs(max_disparity):
                    max_disparity = disparity

                status = "OVER" if disparity > 0 else "UNDER"
                print(f"\n  {cat}:")
                print(f"    Observed:  {observed_prop*100:5.1f}%")
                print(f"    Expected:  {expected_prop*100:5.1f}%")
                print(f"    Disparity: {disparity*100:+5.1f}% ({status}-represented by {abs(disparity_pct):.0f}%)")

        # Chi-square test
        df_chi = len(expected_distribution) - 1
        p_value = 1 - stats.chi2.cdf(chi2_stat, df_chi)

        print("\n" + "-"*60)
        print("STATISTICAL SIGNIFICANCE:")
        print(f"  Chi-square statistic: {chi2_stat:.2f}")
        print(f"  Degrees of freedom: {df_chi}")
        print(f"  P-value: {p_value:.2e}")

        if p_value < 0.001:
            print(f"  *** HIGHLY SIGNIFICANT bias detected (p < 0.001)")
        elif p_value < 0.05:
            print(f"  ** Significant bias detected (p < 0.05)")
        else:
            print(f"  No significant bias detected (p >= 0.05)")

        print("\n" + "="*60)

        return {
            'observed': observed_props.to_dict(),
            'disparities': disparities,
            'chi2_statistic': chi2_stat,
            'p_value': p_value,
            'max_disparity': max_disparity
        }

    return {'observed': observed_props.to_dict()}

# Define expected population distribution (global)
# Based on Fitzpatrick scale distribution worldwide
expected_skin_distribution = {
    'I-II (Very Light)': 0.15,  # ~15% of global population
    'III (Light)': 0.15,        # ~15%
    'IV (Medium)': 0.25,        # ~25%
    'V (Medium-Dark)': 0.25,    # ~25%
    'VI (Dark)': 0.20           # ~20%
}

# Measure bias
skin_tone_bias = measure_representation_bias(
    df,
    attribute='skin_tone',
    expected_distribution=expected_skin_distribution
)

## Part 3: Visualizing Representation Bias

Visualizations make bias immediately apparent.

In [ ]:
def visualize_representation_bias(df_list, labels, attribute='skin_tone',
                                 expected_distribution=None):
    """
    Create comprehensive visualizations of representation bias.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Distribution comparison across bias levels
    ax = axes[0, 0]
    x = np.arange(len(df_list[0][attribute].unique()))
    width = 0.25

    for i, (df_curr, label) in enumerate(zip(df_list, labels)):
        props = df_curr[attribute].value_counts(normalize=True).sort_index()
        ax.bar(x + i*width, props.values * 100, width, label=label, alpha=0.8)

    ax.set_xlabel('Skin Tone Category', fontsize=11)
    ax.set_ylabel('Percentage of Dataset', fontsize=11)
    ax.set_title('Dataset Composition by Bias Level', fontsize=12, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(props.index, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # 2. Observed vs Expected (for high bias dataset)
    ax = axes[0, 1]
    df_main = df_list[0]  # High bias dataset
    observed = df_main[attribute].value_counts(normalize=True).sort_index()

    if expected_distribution:
        categories = observed.index
        observed_vals = [observed[cat] * 100 for cat in categories]
        expected_vals = [expected_distribution.get(cat, 0) * 100 for cat in categories]

        x_pos = np.arange(len(categories))
        width = 0.35

        bars1 = ax.bar(x_pos - width/2, observed_vals, width, label='Dataset (Observed)',
                      color='#e74c3c', alpha=0.8)
        bars2 = ax.bar(x_pos + width/2, expected_vals, width, label='Population (Expected)',
                      color='#2ecc71', alpha=0.8)

        ax.set_xlabel('Skin Tone Category', fontsize=11)
        ax.set_ylabel('Percentage', fontsize=11)
        ax.set_title('Dataset vs Population Distribution', fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(categories, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        # Add disparity annotations
        for i, (obs, exp) in enumerate(zip(observed_vals, expected_vals)):
            disparity = obs - exp
            if abs(disparity) > 2:  # Only annotate significant disparities
                y_pos = max(obs, exp) + 2
                color = 'red' if disparity > 0 else 'blue'
                ax.text(i, y_pos, f"{disparity:+.0f}%", ha='center',
                       fontsize=9, color=color, fontweight='bold')

    # 3. Representation by geography
    ax = axes[1, 0]
    geo_counts = df_main['geography'].value_counts()
    colors_geo = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    wedges, texts, autotexts = ax.pie(geo_counts.values, labels=geo_counts.index,
                                       autopct='%1.1f%%', colors=colors_geo,
                                       startangle=90)
    ax.set_title('Geographic Distribution of Data Sources', fontsize=12, fontweight='bold')
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

    # 4. Sample size by group (critical for fairness evaluation)
    ax = axes[1, 1]
    group_sizes = df_main[attribute].value_counts().sort_index()
    colors_bars = ['#2ecc71' if size > 1000 else '#f39c12' if size > 500 else '#e74c3c'
                   for size in group_sizes.values]
    bars = ax.barh(range(len(group_sizes)), group_sizes.values, color=colors_bars, alpha=0.8)
    ax.set_yticks(range(len(group_sizes)))
    ax.set_yticklabels(group_sizes.index)
    ax.set_xlabel('Sample Size', fontsize=11)
    ax.set_title('Sample Size by Group (Absolute Counts)', fontsize=12, fontweight='bold')
    ax.axvline(x=1000, color='green', linestyle='--', alpha=0.5, label='Adequate (>1000)')
    ax.axvline(x=500, color='orange', linestyle='--', alpha=0.5, label='Marginal (>500)')
    ax.legend()

    # Add sample size annotations
    for i, (cat, size) in enumerate(zip(group_sizes.index, group_sizes.values)):
        ax.text(size + 100, i, f"n={size}", va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('representation_bias_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n✓ Visualization saved as 'representation_bias_analysis.png'")

# Create visualizations
visualize_representation_bias(
    [df_high_bias, df_medium_bias, df_low_bias],
    ['High Bias (Typical)', 'Medium Bias', 'Low Bias (Goal)'],
    attribute='skin_tone',
    expected_distribution=expected_skin_distribution
)

## Part 4: Label Quality and Outcome Bias

Representation bias is just one dimension. We also need to examine:
1. **Label quality**: Are labels equally reliable across groups?
2. **Outcome distribution**: Does disease prevalence differ?
3. **Data quality**: Are measurements equally accurate?

These can introduce additional bias even with balanced representation.

In [ ]:
def analyze_label_quality_bias(df, outcome_col='melanoma',
                              quality_col='biopsy_confirmed',
                              group_col='skin_tone'):
    """
    Analyze whether label quality differs across demographic groups.
    """
    print("="*70)
    print("LABEL QUALITY ANALYSIS")
    print("="*70)
    print("\nLabel quality (biopsy confirmation rate) by group:\n")

    results = []

    for group in sorted(df[group_col].unique()):
        group_data = df[df[group_col] == group]

        # Overall biopsy rate
        biopsy_rate = group_data[quality_col].mean()

        # Biopsy rate for positive cases
        positive_cases = group_data[group_data[outcome_col] == 1]
        if len(positive_cases) > 0:
            positive_biopsy_rate = positive_cases[quality_col].mean()
        else:
            positive_biopsy_rate = 0

        # Biopsy rate for negative cases
        negative_cases = group_data[group_data[outcome_col] == 0]
        if len(negative_cases) > 0:
            negative_biopsy_rate = negative_cases[quality_col].mean()
        else:
            negative_biopsy_rate = 0

        print(f"{group}:")
        print(f"  Overall biopsy confirmation: {biopsy_rate*100:.1f}%")
        print(f"  Melanoma cases confirmed:    {positive_biopsy_rate*100:.1f}%")
        print(f"  Benign cases biopsied:       {negative_biopsy_rate*100:.1f}%")
        print(f"  Sample size: {len(group_data)}\n")

        results.append({
            'group': group,
            'biopsy_rate': biopsy_rate,
            'positive_biopsy_rate': positive_biopsy_rate,
            'n': len(group_data)
        })

    # Statistical test for disparity
    rates = [r['biopsy_rate'] for r in results]
    max_rate = max(rates)
    min_rate = min(rates)
    disparity = max_rate - min_rate

    print("-"*70)
    print(f"Label quality disparity: {disparity*100:.1f} percentage points")
    print(f"  Highest: {max_rate*100:.1f}%")
    print(f"  Lowest:  {min_rate*100:.1f}%")

    if disparity > 0.10:
        print(f"\n⚠️  SIGNIFICANT label quality disparity detected!")
        print(f"     Lower quality labels in underrepresented groups may lead to:")
        print(f"     - Less reliable training signal")
        print(f"     - Amplified bias in model predictions")
    else:
        print(f"\n✓ Label quality is relatively consistent across groups")

    print("="*70)

    return results

# Analyze label quality
label_quality_results = analyze_label_quality_bias(df)

In [ ]:
def analyze_outcome_distribution(df, outcome_col='melanoma', group_col='skin_tone'):
    """
    Analyze disease prevalence across demographic groups.
    """
    print("="*70)
    print("OUTCOME DISTRIBUTION ANALYSIS")
    print("="*70)
    print(f"\nDisease prevalence ({outcome_col}) by group:\n")

    results = []

    for group in sorted(df[group_col].unique()):
        group_data = df[df[group_col] == group]
        prevalence = group_data[outcome_col].mean()
        n_positive = group_data[outcome_col].sum()
        n_total = len(group_data)

        # --- FIX START ---
        # 95% confidence interval
        # binom.interval returns counts (a tuple), so we capture them first
        count_low, count_high = stats.binom.interval(0.95, n_total, prevalence)

        # Then convert counts to proportions
        ci_low = count_low / n_total
        ci_high = count_high / n_total
        # --- FIX END ---

        print(f"{group}:")
        print(f"  Prevalence: {prevalence*100:.2f}% (95% CI: [{ci_low*100:.2f}%, {ci_high*100:.2f}%])")
        print(f"  Cases: {n_positive}/{n_total}\n")

        results.append({
            'group': group,
            'prevalence': prevalence,
            'n_positive': n_positive,
            'n_total': n_total
        })

    # Chi-square test for prevalence differences
    contingency_table = []
    for r in results:
        contingency_table.append([r['n_positive'], r['n_total'] - r['n_positive']])

    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

    print("-"*70)
    print("Statistical test for prevalence differences:")
    print(f"  Chi-square: {chi2:.2f}")
    print(f"  P-value: {p_value:.4f}")

    if p_value < 0.05:
        print(f"  → Prevalence differs significantly across groups (p < 0.05)")
        print(f"\n💡 Note: True prevalence differences are clinically real.")
        print(f"    Models should account for this, not ignore it.")
        print(f"    But beware: differences might also reflect access/detection bias.")
    else:
        print(f"  → No significant prevalence difference detected")

    print("="*70)

    return results

# Analyze outcome distribution
outcome_results = analyze_outcome_distribution(df)

## Part 5: Intersectional Analysis

Bias along multiple dimensions (race × sex × age × geography) can compound.

**Intersectionality**: Some subgroups face multiple forms of marginalization simultaneously.

In [ ]:
def intersectional_bias_analysis(df, attributes=['skin_tone', 'geography']):
    """
    Analyze bias across intersections of multiple protected attributes.
    """
    print("="*70)
    print("INTERSECTIONAL BIAS ANALYSIS")
    print("="*70)
    print(f"\nAnalyzing intersections: {' × '.join(attributes)}\n")

    # Create intersection groups
    df['intersection'] = df[attributes[0]].astype(str)
    for attr in attributes[1:]:
        df['intersection'] += ' × ' + df[attr].astype(str)

    # Count samples per intersection
    intersection_counts = df['intersection'].value_counts()

    print(f"Total unique intersections: {len(intersection_counts)}")
    print(f"\nTop 10 largest groups:")
    for i, (group, count) in enumerate(intersection_counts.head(10).items(), 1):
        pct = count / len(df) * 100
        print(f"  {i:2d}. {group:50s} n={count:4d} ({pct:4.1f}%)")

    print(f"\nBottom 10 smallest groups:")
    for i, (group, count) in enumerate(intersection_counts.tail(10).items(), 1):
        pct = count / len(df) * 100
        status = "⚠️ VERY SMALL" if count < 100 else "⚠️ Small" if count < 500 else ""
        print(f"  {i:2d}. {group:50s} n={count:4d} ({pct:4.1f}%) {status}")

    # Sample size adequacy
    adequate = (intersection_counts >= 1000).sum()
    marginal = ((intersection_counts >= 500) & (intersection_counts < 1000)).sum()
    inadequate = (intersection_counts < 500).sum()

    print("\n" + "-"*70)
    print("Sample Size Adequacy for Fairness Evaluation:")
    print(f"  Adequate (n ≥ 1000):   {adequate:2d} groups ({adequate/len(intersection_counts)*100:.0f}%)")
    print(f"  Marginal (500-999):    {marginal:2d} groups ({marginal/len(intersection_counts)*100:.0f}%)")
    print(f"  Inadequate (n < 500):  {inadequate:2d} groups ({inadequate/len(intersection_counts)*100:.0f}%)")

    if inadequate > len(intersection_counts) * 0.3:
        print(f"\n⚠️  WARNING: {inadequate} intersectional groups have insufficient samples!")
        print(f"     Fairness evaluation for these groups will have low statistical power.")
        print(f"     Consider:")
        print(f"     - Collecting more data from underrepresented intersections")
        print(f"     - Collapsing some categories for analysis")
        print(f"     - Focusing fairness efforts on largest disparities")

    print("="*70)

    # Clean up
    df.drop('intersection', axis=1, inplace=True)

    return intersection_counts

# Perform intersectional analysis
intersection_counts = intersectional_bias_analysis(df, ['skin_tone', 'geography'])

## Key Takeaways

### What We Learned

1. **Representation bias is pervasive**: Real medical datasets often show 90% majority group, 10% minority group - matching historical patterns in dermatology, radiology, and other domains.

2. **Statistical significance matters**: Chi-square tests confirm when observed distributions differ significantly from expected population distributions. P-values < 0.001 indicate severe bias requiring mitigation.

3. **Multiple dimensions of bias**:
   - **Sample size**: Absolute counts matter for statistical power
   - **Label quality**: Biopsy confirmation rates vary by group
   - **Data quality**: Image quality lower for underrepresented groups
   - **Outcome distribution**: True prevalence may differ (biological + social factors)

4. **Intersectionality compounds bias**: Groups at intersection of multiple protected attributes (e.g., dark skin + non-Western geography) are most severely underrepresented.

5. **Sample size requirements**:
   - n > 1000 per group: Adequate for fairness evaluation
   - n = 500-1000: Marginal power
   - n < 500: Insufficient for reliable fairness metrics

### Implications for Fair AI

- **Measure before you mitigate**: Can't fix bias you haven't measured
- **Report transparently**: Stakeholders need to know dataset limitations
- **Stratified evaluation required**: Overall metrics hide disparities
- **Data collection matters**: Technical fixes can't fully compensate for fundamentally biased data

---

## Exercises

1. **Chi-square by hand**: Calculate chi-square statistic manually for a small dataset to understand the math.

2. **Geographic bias**: Analyze how data collection geography correlates with representation bias.

3. **Sampling strategy**: Design an oversampling strategy to achieve balanced representation. What are the trade-offs?

4. **Real-world audit**: Find a public medical imaging dataset and perform representation bias analysis. What do you find?

5. **Label quality simulation**: Modify the simulation to introduce label noise that differs by group. How does this affect model performance?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 9: Fairness and Bias)*  
*Next: Notebook 9.2 - Fairness Metrics*

---